# P00 — panel preparation for the four models (M01, M02, M16, M34)
Run it top to bottom. It builds the panel ONCE with what these four models read — every ring, year and season, every row located
in the sub-watershed shapefile and coded (the location rule), repeated rows dropped whole, the rows of other sub-watersheds kept
coded so every model leaves them out — and nothing for any other model: no ground linkage, no benchmark surrogates, no dose,
no land use, no readiness or package checks of the other 41 models. Settings are in **one** cell (`P00_Settings`).
The panel and everything else are written to `<data>\output_4Models` (the full pipeline's `<data>\output` is never touched).

| # | cell | what it does | engine functions |
|---|---|---|---|
| 1 | `P00_Settings` | every choice for the preparation, in ONE place | -- |
| 2 | `P01_File_Inventory_and_Audit` | which export files exist, per year and season; naming styles | discover_input_files, parse_filename |
| 3 | `P02_Column_Harmonization_Check` | column-name dry run on a stratified sample | harmonize_columns |
| 4 | `P00_Validate_or_Skip` | keep a complete, current panel instead of rebuilding | final_panel_is_valid |
| 5 | `P03_Ingest_Harmonize_and_Assign_Pixel_UIDs` | PASS A: ingest, harmonise, pixel ids, every row located in the shapefile | run_pass_a |
| 6 | `P05_Fund_Release_Treatment_Timing` | your fund workbook: each sub-watershed's first treated season (the models apply it when they run) | _fund.build_fund_tables |
| 7 | `P04_Assemble_Ordered_Panel` | PASS B: one panel, cross-file dedup (repeated rows dropped whole), pixel linking | run_pass_b |
| 8 | `P06_Panel_Integrity_and_Readiness_Report` | integrity, duplicates confirmed, validity verdict | streaming_panel_stats, confirm_panel_duplicates |
| 9 | `P00_Design_for_Every_Model` | the design defaults the four models inherit, the timing per sub-watershed, season report | set_scenario, resolve_design |
| 10 | `P09_Estimator_Variable_Files` | per-variable files (fast loading), baselines and pixel counts | build_estimator_files |
| 11 | `P10_Model_Readiness` | whether this panel completes / limits each of the four models | model_readiness |
| 12 | `P12_Prebuilt_Packages_Verify` | install and verify the four models' packages (Python: pyfixest; R: fixest, HonestDiD) and report the out-of-core engines (Dask, Spark) | verify_prebuilt_routes, verify_r_models |


**REWARD DiD — the four-model pipeline (P00, M01, M02, M16, M34), engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output = `<data>\output_4Models`, temp, panel, results and estimator files derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Whether the data support the four models: P10. Pre-built packages: `python_prebuilt/` and the R routes (`R/lib` -- the library of the R pipeline in this bundle's `R/` folder). Details: the bundle's `README.md` and `docs/`.

In [ ]:
# ---- CELL 0: libraries (v17.3) -- run first; prints what to pip-install if anything is missing ----
import importlib as _il, importlib.util, sys as _sys
_REQ = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
        "psutil": "psutil", "tqdm": "tqdm", "joblib": "joblib", "openpyxl": "openpyxl"}
_OPT = {"torch": "GPU demeaning (CPU path without it)", "ipywidgets": "progress-bar widget (text bar without it)",
        "dask": "out of core beyond 98 % of the RAM, the first engine (pip install \"dask[distributed]\")",
        "pyspark": "out of core beyond 98 % of the RAM, the second engine (pip install pyspark; Java 17+)"}
_miss = [pipn for imp, pipn in _REQ.items() if _il.util.find_spec(imp) is None]
print("[INFO]    python", _sys.version.split()[0], "| missing REQUIRED:", _miss or "none",
      "| optional absent:", [k for k in _OPT if _il.util.find_spec(k) is None] or "none")
if _miss: print("[WARNING] run in a terminal:  pip install " + " ".join(_miss) + "   (or leave INSTALL_PACKAGES = True in P00_Settings: P12 installs every package that is missing -- wheel first, then source -- and confirms all of them)")


In [ ]:
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "P" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_prep_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_prep_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    P = _il.import_module("_prep_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _prep_common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
# ---- CELL 0: ENVIRONMENT GUARD (run this FIRST, every time) ----
# Your last run executed inside ArcGIS Pro's Python 3.7 by mistake. Its site-packages live in
# "Program Files" (write-protected), so "!pip install" failed and half-overwrote pyarrow,
# breaking it. This cell tells you EXACTLY which Python the kernel is using and refuses to
# continue if it is the wrong one -- so the failure is loud, immediate, and explained.
import sys, platform
exe = sys.executable
print(f"[INFO]    Python executable : {exe}")
print(f"[INFO]    Python version    : {platform.python_version()}")
_bad = "arcgis" in exe.lower() or "program files" in exe.lower()
_old = sys.version_info < (3, 9)
if _bad or _old:
    print("[FAILED]  WRONG PYTHON ENVIRONMENT.")
    if _bad: print("[FAILED]  This kernel is ArcGIS Pro's Python (write-protected, old packages).")
    if _old: print(f"[FAILED]  Python {platform.python_version()} is too old; this pipeline needs >= 3.9.")
    print("[INFO]    FIX (30 seconds): Jupyter menu -> Kernel -> Change Kernel ->")
    print("[INFO]         choose  'Python [conda env:anaconda3]'  (the one your P01-P06 runs used)")
    print("[INFO]    then re-run this cell. Do NOT pip-install into the ArcGIS environment.")
    raise SystemExit("switch kernel, then re-run from CELL 0")
_missing = []
for pkg in ("pandas", "numpy", "scipy", "pyarrow", "openpyxl"):
    try:
        m = __import__(pkg); print(f"[OK]      {pkg:10s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        _missing.append(pkg); print(f"[FAILED]  {pkg:10s} -> {type(e).__name__}: {str(e)[:70]}")
if _missing:
    print(f"[INFO]    install into THIS environment from an Anaconda Prompt:")
    print(f"[INFO]         conda activate anaconda3")
    print(f"[INFO]         pip install {' '.join(_missing)}")
    print("[INFO]    then Kernel -> Restart, and re-run from CELL 0.")
    raise SystemExit("missing packages -- see above")
print("[OK]      environment is correct -- proceed to STEP 1")

## P00_Settings
Every choice, once.

In [ ]:
# ===== P00_Settings.ipynb -- EVERY choice for the whole preparation, in ONE place (v20.38) =====
# ---- THIS PROJECT (v20.58): the panel for the four models M01, M02, M16, M34 only (_paths.PIPELINE_MODELS) -> <data>\output_4Models
# ---- v20.57: THE DESIGN IS CHOSEN IN EACH MODEL (its CELL 1), WHEN THE MODEL RUNS -- not here. P00 builds the panel ONCE with every
#      row (all rings, all years, all seasons, every sub-watershed and fragment, each row coded) and never has to be re-run for a
#      design choice. The values below are only the DEFAULTS P00 saves (did_scenario.json) for an option a model leaves unset (None),
#      and the design P00's own reports use (DESIGN_RECOMMENDATION.md, the season report). A model's CELL 1 always wins.
DESIGN_MODE    = "recommended"  # "recommended": an option set to "data" is chosen FROM THE DATA (export breaks, fill years, spillover into
                                #   inner rings -- never from the effect; DESIGN_RECOMMENDATION.md says why) | "manual": "data" = every ring / year
TREATMENT_TIMING = "fund"       # v20.57: "fund" (your fund workbook: each sub-watershed's first treated season) | "registry" (sites.csv) | "fixed"
TREATMENT_YEAR = 2022           # "fixed" timing, and the fall-back for a sub-watershed the fund file / registry does not date
FUND_START_RULE = "backcast"    # v20.57: "backcast" | "share" | "file_start" (FUND_TIMING_AND_DOSE.md)
CONTROL_ZONES  = "data"         # "data" | "1-5" | "1-3" ... (the TREATMENT AREA is buffer 0 -- the core of each sub-watershed)
EXCLUDE_TRANSITION_YEAR = False # True = the first treated year of each series leaves the sample (robustness)
PRE_YEARS  = "data"             # "data" | "all" | a number of years | 2015 (a calendar year = the FIRST pre year)
POST_YEARS = "data"
COHORT_OFFSET  = 0
OVERLAP_ROWS   = "drop"         # v20.55 (your option): pooled panels -- rows of pixels that are TREATED in another sub-watershed but lie in
                                #   a control ring of this one, and rows repeating a pixel already in the sample: "drop" | "keep"
FRAGMENT_RULE  = "drop"         # v20.57 (your rule): only the MAJOR sub-watershed data -- fragments of other sub-watersheds inside the exports
                                #   and minor sub-watersheds of the panel are left out | "keep". The panel keeps every row, coded (fragment)
CLUSTER        = "site"         # the SUB-WATERSHED is the cluster (20 when pooled); ONE sub-watershed -> years
POOLED_FE      = "site_period"  # several sub-watersheds: sub-watershed x year x season effects | "period" (one shared effect). R: POOLED_FE
SEASONS        = "all"          # YOUR RULE: annual composite + Kharif / Rabi / Zaid together (year AND season variation) |
                                #   "seasonal" | "yearly" | "Rabi", "Rabi+Zaid" | "auto" = the data decide
UNIT_FE        = "pixel_season" # one fixed effect per pixel x season series (+ year x season)
EXCLUDE_GAPFILLED = True        # rows the exporter filled from history are not observations | False keeps them (tag _withGapFilled)
OUTCOME_SCREEN    = "drop"      # v20.59: "drop" (a year-season constant across pixels -- a fill value -- or with collapsed coverage leaves the
                                #   model; its evidence: OUTCOME_SCREEN_<outcome>.csv beside the results) | "keep" (reported and KEPT; results tagged
                                #   _screenKept) | "off". A v20.58 run stopped here ("41 year-seasons are NOT pixel data ... Re-export it")
# covariates are chosen in EACH MODEL notebook (COVARIATES in its CELL 1) -- not here
# ---- preparation ----
P.PIXEL_OVERLAP_MIN = 0.65      # links exports on a grid shifted a few metres (your 2025 exports: median 3.0 m)
# v20.58: the duplicate, pixel-linkage and negative-covariate rules are set HERE (until v20.57 in the PASS B cell -- AFTER P00_Validate_or_Skip
#   had compared the panel on disk with the defaults, so a change there was silently ignored while the old panel was kept). Another value
#   than the panel on disk was built with -> the panel is rebuilt.
P.NEAR_DUPLICATE_PIXELS = True      # pixels whose 10 m footprints overlap >= PIXEL_OVERLAP_MIN are ONE pixel | False = only identical coordinates
P.PIXEL_SIZE_M          = 10.0      # pixel side; the run prints the grid spacing measured per export
P.DEDUP_PRIORITY        = "newer"   # which REPEATED row (the same pixel, year and season in two exports) is kept: "newer" = the newer
                                    #   export, then the more complete row | "complete" = the more complete row, then the newer export
P.DEDUP_FILL_FROM_DUPLICATES = False  # v20.58 (your rule): a repeated row is DROPPED WHOLE -- none of its values enters the panel, not even
                                    #   into a gap of the kept row | True = fill the kept row's gaps from it (exports split by variable:
                                    #   one file NDVI, another LAI of the same pixel-period; until v20.57 always on)
P.POST_FROM_EXPORT_TREAT = True   # v20.59 (your rule): the panel's post / pre come from the exports' Treat flag (1 = post, 0 = pre); a row without a
                                  #   usable flag takes the rule Year >= TREATMENT_YEAR. treat / control / pre / post / did are in the panel; every model
                                  #   applies ITS OWN design when it runs and says on how many rows it differs ("DESIGN vs PANEL"). False = the rule (v20.58)
P.NEGATIVE_COVARIATE_RULE   = {"Rain": "zero", "Tmax": "zero", "Tmean": "zero", "Tmin": "zero"}   # v20.30: negative Rain / temperatures -> 0;
P.ALLOW_NEGATIVE_COVARIATES = False # no-data fill values (<= -100) and the -10 C clamp -> missing | "missing" blocks negatives | True = no barrier
N_WORKERS = None                # PASS A worker processes: None = every logical core the platform allows (60 on Windows; was 32)
OUTCOMES  = None                # None = every variable; or e.g. ["NDVI", "LAI"]
OVERWRITE = False               # per-variable files: rebuild even when current
# ---- which merged modules run (True = part of the one run) ----
RUN_ESTIMATOR_FILES = True      # P09 (+ baselines and pixel counts of every variable)
RUN_READINESS       = True      # P10
VERIFY_PACKAGES     = True      # P12: the four models' packages (Python: pyfixest; R: fixest, HonestDiD), verified before they are used
INSTALL_PACKAGES    = True      # P12: pip-install them if missing (needs internet; skipped cleanly without it)
R_SCRIPT            = None      # Rscript path; None = found automatically (PATH, R_HOME, C:\\Program Files\\R\\R-*)
INSTALL_R_PACKAGES  = True      # P12: install the R packages of the four models' R routes (CRAN)
import _common as C
C.start_cell_log("P00")        # every message of P00, from here on, also goes to <OUTPUT_DIR>\cell_logs
print(f"[OK]      P00_Settings: panel preparation settings (pixel overlap {P.PIXEL_OVERLAP_MIN}; repeated rows: the {P.DEDUP_PRIORITY} row, "
      f"{'gaps filled from the others' if P.DEDUP_FILL_FROM_DUPLICATES else 'the others dropped whole'}); the DESIGN defaults saved for the "
      f"models: timing {TREATMENT_TIMING}, rings {CONTROL_ZONES}, seasons {SEASONS}, fragments {FRAGMENT_RULE} -- each model's CELL 1 sets its own")


## P01_File_Inventory_and_Audit
which export files exist, per year and season; naming styles  
*(formerly `P01_File_Inventory_and_Audit.ipynb`)*

In [ ]:
# ===== P01_File_Inventory_and_Audit.ipynb (merged into P00) =====
# ---- ROBUST IMPORT BOOTSTRAP ----
import sys, os, glob as _glob
def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d); cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)): return c
    return None
_dir = _locate("_prep_common.py")
if _dir is None:
    print("[FAILED]  Could not find _prep_common.py")
    print(f"[INFO]    Jupyter working directory: {os.getcwd()}")
    print("[INFO]    Open from inside DIDRDP_ALLRunDIDv13, or set manually:")
    print("[INFO]      sys.path.insert(0, r'C:\\path\\to\\DIDRDP_ALLRunDIDv13\\01_Panel_Preparation')")
    raise ModuleNotFoundError("_prep_common.py not found")
if _dir not in sys.path: sys.path.insert(0, _dir)
print(f"[OK]      found _prep_common.py in: {_dir}")

import numpy as np, pandas as pd, json, traceback, glob, random
import _prep_common as P
import _common as _CL; _CL.start_cell_log("P00")   # v20.36: every message also goes to <OUTPUT_DIR>\cell_logs
P.require_engine("20.44")     # minimum engine version; a newer engine is fine     # stop immediately on a mixed/cached engine
P.validate_output_dir()
P.autotune_prep()
P.step(1,"file inventory & audit")
_excl=[os.path.abspath(x) for x in (P.OUTPUT_DIR,P.TEMP_DIR)]      # FIX v13
P.info(f"missing-value policy: NaN and exact 0 = no-data in {len(P.ZERO_RULE_VARS)} variables" + (f" except {P.ZERO_RULE_EXCEPT}" if P.ZERO_RULE_EXCEPT else "") + "; rows with no usable outcome are dropped (P.ZERO_AS_MISSING / P.ZERO_RULE_EXCEPT to change)")
import _paths as _PP; P.ok('paths in force (INPUT_DIR set in _paths.py, or P.set_paths(...)):'); print(_PP.describe(P.ACTIVE_PATHS or _PP.derive()))
files,_nx,_nseen=P.discover_input_files(P.INPUT_DIR,P.OUTPUT_DIR,P.TEMP_DIR,verbose=True)   # v20.14: every format, any name
P.ok(f"{len(files):,} candidate files under {P.INPUT_DIR} (output folder excluded)")
if not files: P.fail("ZERO files -- check INPUT_DIR in _paths.py"); raise SystemExit

rows,bad=[],[]
for f in files:
    m=P.parse_filename(f)
    if m: rows.append({**m,"file":os.path.basename(f),"ext":os.path.splitext(f)[1],"path":f})
    else: bad.append(os.path.basename(f))
inv=pd.DataFrame(rows)
P.ok(f"{len(inv):,} parsed, {len(bad)} unparseable")
if len(inv):
    print("\nfiles per Year x Season:")
    print(inv.pivot_table(index="Year",columns="Season",values="file",aggfunc="count").fillna(0).astype(int).to_string())
    sites=inv["site_from_name"].dropna().unique().tolist()
    if sites: P.ok(f"site name embedded in {int(inv['site_from_name'].notna().sum()):,} filenames: {sites}")
    g=inv.groupby(["Year","Season"])["site_from_name"].agg(named=lambda s:s.notna().sum(),
                                                            unnamed=lambda s:s.isna().sum())
    both=g[(g["named"]>0)&(g["unnamed"]>0)]
    if len(both):
        P.warn(f"{len(both)} Year/Season block(s) contain BOTH naming styles -- these produce")
        P.warn("cross-file duplicate pixels. STEP 4 resolves them (the v13 fix).")
        print(both.to_string())
if bad: P.warn(f"unparseable (expected for non-panel files): {bad[:8]}")
inv.to_csv(os.path.join(P.OUTPUT_DIR,"file_inventory.csv"),index=False)
P.done(1, next_task="STEP 2 -- column-harmonization dry run (STRATIFIED sample -- v13 fix)")


## P02_Column_Harmonization_Check
column-name dry run on a stratified sample  
*(formerly `P02_Column_Harmonization_Check.ipynb`)*

In [ ]:
# ===== P02_Column_Harmonization_Check.ipynb (merged into P00) =====
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "P" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_prep_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_prep_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    P = _il.import_module("_prep_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _prep_common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
P.step(2,"column-harmonization dry run (STRATIFIED sample -- v13 fix)")
def _ncols(f):
    try: return P.read_export_raw(f, nrows=0).shape[1]      # v20.55: every export format the pipeline reads (was: csv else parquet -- an .xlsx failed here)
    except Exception: return -1
cand=inv["path"].tolist()
newest=sorted(cand,key=lambda f: os.path.getmtime(f),reverse=True)[:8]
by_width={}
for f in cand[:200]:
    w=_ncols(f)
    by_width.setdefault(w,f)
random.seed(0)
sample=list(dict.fromkeys(newest+list(by_width.values())+random.sample(cand,min(8,len(cand)))))
P.ok(f"stratified sample: {len(sample)} files (newest 8 + one per column-width + random)")
P.info(f"column widths present: {sorted(by_width)}  <- a random sample can miss the widest/newest")

unresolved,per_file=[],[]
for f in sample:
    try:
        df=P.read_export_raw(f, nrows=5)
        mp=P.harmonize_columns(list(df.columns),fname=os.path.basename(f),unresolved_log=unresolved)
        per_file.append({"file":os.path.basename(f),"n_cols":df.shape[1],
                         "n_renamed":sum(1 for k,v in mp.items() if k!=v)})
    except Exception as e: P.warn(f"{os.path.basename(f)}: {e}")
print(pd.DataFrame(per_file).to_string(index=False))
P.info("n_renamed=0 is GOOD -- the columns already match the canonical names.")
if unresolved:
    u=pd.DataFrame(unresolved,columns=["file","column"]).drop_duplicates("column")
    P.warn(f"{len(u)} column name(s) not confidently mapped:"); print(u.to_string(index=False))
    P.info("Add them to ALIASES or CANONICAL in _prep_common.py, then re-run.")
else:
    P.ok("every column mapped cleanly across ALL column-widths present")
json.dump({"unresolved":unresolved,"n_files_checked":len(per_file)},
          open(os.path.join(P.OUTPUT_DIR,"harmonization_check.json"),"w"),indent=2)
P.done(2, next_task="STEP 3 -- PASS A -- ingest, harmonize, stable pixel UIDs, QC, per-file")

## P00_Validate_or_Skip
keep a complete, current panel instead of rebuilding  
*(formerly `P00_Validate_or_Skip.ipynb`)*

In [ ]:
# ===== P00_Validate_or_Skip.ipynb (merged into P00) =====
# ---- v17.6: validate-or-skip -- an existing, complete final panel is NOT rebuilt unless FORCE_REBUILD = True ----
FORCE_REBUILD = False
P.SKIP_PANEL_BUILD = (not FORCE_REBUILD) and P.final_panel_is_valid()
if P.SKIP_PANEL_BUILD: P.info('PASS A / PASS B skipped -- the panel on disk already matches FINAL_PANEL_COLUMNS')


## P03_Ingest_Harmonize_and_Assign_Pixel_UIDs
PASS A: ingest, harmonise, pixel ids, sub-watershed tagging  
*(formerly `P03_Ingest_Harmonize_and_Assign_Pixel_UIDs.ipynb`)*

In [ ]:
# ===== P03_Ingest_Harmonize_and_Assign_Pixel_UIDs.ipynb (merged into P00) =====
if getattr(P, 'SKIP_PANEL_BUILD', False):
    P.info('skipped (final panel valid; FORCE_REBUILD=True to rebuild)')
else:
    # v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
    if "P" not in globals():
        import os as _os, sys as _sys, importlib as _il
        _d = _os.path.abspath(_os.getcwd()); _found = None
        for _ in range(6):
            for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
                if _os.path.exists(_os.path.join(_c, "_prep_common.py")): _found = _c; break
            if _found or _os.path.dirname(_d) == _d: break
            _d = _os.path.dirname(_d)
        if _found is None: raise RuntimeError("_prep_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
        if _found not in _sys.path: _sys.path.insert(0, _found)
        P = _il.import_module("_prep_common"); import numpy as np, pandas as pd, os, json
        print(f"[INFO]    auto-bootstrap: _prep_common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
    P.step(3,"PASS A -- ingest, harmonize, stable pixel UIDs, QC, per-file dedup")
    P.info("The long one. Progress prints every 200 files.")
    if P.pq is None:
        P.fail("pyarrow is not importable in THIS kernel -- almost always the wrong Python environment.")
        P.info("Go back to CELL 0 (environment guard); it will tell you which kernel to switch to.")
        raise SystemExit("wrong environment")
    P.ok(f"pyarrow ready | {os.cpu_count()} logical CPUs available for PASS A")
    try:
        P.KNOWN_SUBWSHED_NAMES=P.load_subwshed_crosswalk(P.SUBWSHED_CROSSWALK_PATH)["Sub Watershed Name"].tolist()
        P.ok(f"crosswalk: {len(P.KNOWN_SUBWSHED_NAMES)} sub-watershed names")
    except Exception as e:
        P.warn(f"crosswalk unavailable ({e}) -- site_name will be unresolved")
    try:
        # v16: parallel PASS A. Your box has 64 logical CPUs; 32 workers x ~1 GB each fits easily in 512 GB.
        # v20.38: N_WORKERS comes from P00_Settings (None = every core the platform allows; was hard-coded 32)
        # ---- v20.28: every row is located in the 20-SWS shapefile (data/sites/SWSs20_KarnatakaAll5k) BEFORE de-duplication:
        # its SWSiD_All / SUBWSHED name / folder name is confirmed, corrected or assigned -> site_id, sws_name, site_check
        P.SITE_GEOMETRY_CHECK = True      # False = trust the id in the data (previous behaviour)
        P.BUFF_FROM_GEOMETRY  = False     # True = buff_km always from the polygon ring
        # ---- v20.30: MEMORY FIRST -- PASS A hands its blocks to PASS B in RAM (no shard files) while they fit below
        # 95 % of total RAM; beyond that it spills to shard files and continues exactly as before. P03 -> P04 always
        # use shard files (they run in different kernels).
        P.IN_MEMORY_BLOCKS = True
        import _hardware as _H; print('[INFO]   ', _H.memory_report())
        registry,unresolved_cols,parse_errors,dedup_conflicts,shard_paths = P.run_pass_a(
            P.INPUT_DIR,P.TEMP_DIR,P.OUTPUT_DIR,n_workers=N_WORKERS)
        P.ok(f"PASS A COMPLETE: {len(registry.uids):,} unique pixels, {len(shard_paths)} shards")
        (P.warn if unresolved_cols else P.ok)(f"unresolved columns: {len(unresolved_cols)}")
        _n_err = P._n_file_errors(parse_errors) if hasattr(P, "_n_file_errors") else len(parse_errors)
        (P.warn if _n_err else P.ok)(f"file errors        : {_n_err}" + (f"  (+ {len(parse_errors) - _n_err} notes: outcome columns absent from a file were filled NaN -- see file_errors.json)" if len(parse_errors) > _n_err else ""))
        P.info(f"within-file duplicate groups: {len(dedup_conflicts)} "
               f"(CROSS-file duplicates are handled in STEP 4)")
    except Exception as e:
        P.fail(f"PASS A failed: {e}"); traceback.print_exc(); raise
    P.done(3, next_task="STEP 4 -- fund-release timing (your fund workbook, per sub-watershed)")

## P05_Fund_Release_Treatment_Timing
your fund workbook -> each sub-watershed's first treated season (written to results/FUND; the four models apply it when they run -- no dose: none of them uses it)  
*(formerly `P05_Fund_Release_Treatment_Timing_and_Dose.ipynb`)*

In [ ]:
# ===== P05_Fund_Release_Treatment_Timing.ipynb (merged into P00) =====
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "P" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_prep_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_prep_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    P = _il.import_module("_prep_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _prep_common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
P.step(4,"fund-release timing (the four models take each sub-watershed's first treated season from your fund workbook when they run)")
dose_table=None; xw=None      # v20.58 (the four-model project): no dose is merged into the panel -- none of M01, M02, M16, M34 uses it
# ---- v20.57: YOUR FUND WORKBOOK AS EACH SUB-WATERSHED'S TREATMENT TIMING AND DOSE (_fund.py) -- computed AGAIN by every model when
# it runs (TREATMENT_TIMING = "fund" in its CELL 1; the dose of every timing), so a new start rule or a revised workbook takes effect
# without rebuilding the panel. Written here so you can read it before any model runs:
#   FUND_TIMING.csv          per sub-watershed: area, target, the first month, the back-cast start, the FIRST TREATED SEASON, the cohort
#   FUND_SEASON_DOSE.csv     per sub-watershed x Year x Season: amount released by the end of the previous season, per ha, share of target
#   FUND_TIMING_AND_DOSE.md  the same as a table, with the rule
try:
    import _fund as F
    _yrs = list(range(2010, 2036))
    _tim, _dose, _ser = F.build_fund_tables(P.FUND_RELEASE_PATH, _yrs, out_dir=os.path.join(P.OUTPUT_DIR, "results", "FUND"))
    print(_tim[["site_id", "sws_name", "area_ha", "first_month", "amount_first_month", "backcast_start", "first_treated_label", "cohort_annual"]].to_string(index=False))
    P.ok("the fund timing and dose (v20.57) -> results/FUND/ -- the models apply them at run time (TREATMENT_TIMING = 'fund')")
except Exception as e:
    P.warn(f"fund timing / dose tables not written ({type(e).__name__}: {e}) -- a model with TREATMENT_TIMING = 'fund' falls back to "
           f"the registry years and says so")
P.done(4, next_task="STEP 5 -- PASS B -- ordered panel + CROSS-FILE dedup (repeated rows dropped whole)")

## P04_Assemble_Ordered_Panel
PASS B: one panel, cross-file dedup (repeated rows dropped whole), pixel linking, DiD columns  
*(formerly `P04_Assemble_Ordered_Panel.ipynb`)*

In [ ]:
# ===== P04_Assemble_Ordered_Panel.ipynb (merged into P00) =====
if getattr(P, 'SKIP_PANEL_BUILD', False):
    P.info('skipped (final panel valid; FORCE_REBUILD=True to rebuild)')
else:
    # v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
    if "P" not in globals():
        import os as _os, sys as _sys, importlib as _il
        _d = _os.path.abspath(_os.getcwd()); _found = None
        for _ in range(6):
            for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
                if _os.path.exists(_os.path.join(_c, "_prep_common.py")): _found = _c; break
            if _found or _os.path.dirname(_d) == _d: break
            _d = _os.path.dirname(_d)
        if _found is None: raise RuntimeError("_prep_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
        if _found not in _sys.path: _sys.path.insert(0, _found)
        P = _il.import_module("_prep_common"); import numpy as np, pandas as pd, os, json
        print(f"[INFO]    auto-bootstrap: _prep_common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
    P.step(5,"PASS B -- ordered panel + CROSS-FILE dedup (repeated rows dropped whole) + DiD columns, ALL per shard (1B-row safe)")
    if 'shard_paths' not in dir():
        P.fail("shard_paths is not defined -> STEP 3 (PASS A) did not complete. Fix STEP 3 first; do not skip ahead.")
        raise SystemExit("STEP 3 incomplete")
    try:
        # ---- v20.25: pixels whose 10 m footprints overlap by >= PIXEL_OVERLAP_MIN (or coincide) are ONE pixel ---------
        # Two exports can sample the same physical pixel on grids a metre or two apart; without this they get two ids,
        # are counted twice in a block, and look like different pixels across years. PASS B maps each such pixel onto one
        # canonical pixel, then keeps one row per (site, pixel, Year, Season): the newer export's row AS IT IS (v20.58: the repeated
        # rows are dropped whole -- P.DEDUP_FILL_FROM_DUPLICATES = True in P00_Settings fills the kept row's gaps from them).
        # v20.55: the threshold is set ONCE, in P00_Settings (P.PIXEL_OVERLAP_MIN = 0.65 since v20.38). This cell used to
        # reset it to 0.80 here, so the settings cell had no effect: your 2025 exports (grid shifted ~3 m, overlap 0.70)
        # were never linked in Python, 2025 became an "export break" and left the post period -- while R linked them.
        # v20.58: the same for NEAR_DUPLICATE_PIXELS, PIXEL_SIZE_M, DEDUP_PRIORITY, DEDUP_FILL_FROM_DUPLICATES and the negative-covariate
        # barrier (NEGATIVE_COVARIATE_RULE, ALLOW_NEGATIVE_COVARIATES): set in P00_Settings only (here they came AFTER the validate-or-skip
        # step, so a change was ignored while the old panel was kept).
        # ---- v20.30: NO NEGATIVE COVARIATES (prepared here, at panel level) ----------------------------------
        # negative Rain / Tmax / Tmean / Tmin -> 0 ("zero"); no-data fill values (<= -100) and the exporter's -10 C temperature
        # clamp -> missing (never a fake 0). "missing" blocks negatives instead; "keep" or ALLOW_NEGATIVE_COVARIATES = True removes
        # the barrier (the panel is then rebuilt). Outcomes (NDVI etc.) are never touched. Report: negative_covariates_report.csv
        P.info(f"pixel linkage: footprints overlapping >= {P.PIXEL_OVERLAP_MIN:.0%} are one pixel ({'on' if P.NEAR_DUPLICATE_PIXELS else 'OFF'}, "
               f"{P.PIXEL_SIZE_M:g} m pixels); repeated rows: the {P.DEDUP_PRIORITY} row wins, "
               f"{'its gaps filled from the others' if P.DEDUP_FILL_FROM_DUPLICATES else 'the others dropped whole'}; negative covariates: "
               f"{'no barrier' if P.ALLOW_NEGATIVE_COVARIATES else P.NEGATIVE_COVARIATE_RULE} (all from P00_Settings)")
        final_path=P.run_pass_b(shard_paths,P.OUTPUT_DIR,dose_table=dose_table,crosswalk=xw)
        P.ok(f"panel written -> {final_path}")
        man=P.build_manifest(final_path)
        print(json.dumps({k:v for k,v in man.items() if k!="row_order_check"},indent=2,default=str))
        P.ok(f"final panel schema: {len(P.FINAL_PANEL_SCHEMA)} columns, compact dtypes (~170 B/row); "
             f"{len(P.DROPPED_FROM_PANEL)} audit/duplicate/raw columns dropped")
        P.info("PASS B refuses to write a shard that still has a duplicate pixel-Year-Season -- the guarantee you asked for.")
    except Exception as e:
        P.fail(f"PASS B failed: {e}"); traceback.print_exc(); raise
    P.done(5, next_task="STEP 6 -- panel integrity, validity verdict & readiness -- STREAMED (n")

## P06_Panel_Integrity_and_Readiness_Report
integrity, validity verdict  
*(formerly `P06_Panel_Integrity_and_Readiness_Report.ipynb`)*

In [ ]:
# ===== P06_Panel_Integrity_and_Readiness_Report.ipynb (merged into P00) =====
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py like CELL 1 does)
if "P" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_prep_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_prep_common.py not found -- run CELL 1 (import bootstrap) first, or open Jupyter from the DIDRDP_ALLRunDID_FINAL folder")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    P = _il.import_module("_prep_common"); import numpy as np, pandas as pd, os, json
    print(f"[INFO]    auto-bootstrap: _prep_common imported from {{_found}} (CELL 1 was not run in this kernel -- variables it defines may be missing)")
P.step(6,"panel integrity, validity verdict & readiness -- STREAMED (never loads the whole panel)")
try:
    KEY=["buff_km","Season","Year","treatment","control","pre","post","did_term","subwshed_id","site_name","District","LandUse","first_treat_agri_year"]
    OUT=[c for c in ["NDVI","SAVI","EVI","LAI","LSWI","NDWI","NDMI","NDRE","AGB","RUSLE","Rain","Tmax","Tmean","Tmin","ESI","WSSI","WSI","SMDI","VCI","TCI","VHI"]]
    import pyarrow.parquet as pq
    have=set(pq.ParquetFile(P.FINAL_PANEL).schema_arrow.names)
    st=P.streaming_panel_stats(P.FINAL_PANEL,key_cols=[k for k in KEY if k in have],outcome_cols=[o for o in OUT if o in have],pixel_sample_mod=100)
    print(f"rows {st['n_rows']:,} | pixel sample {st['pixel_sample_size']:,} (1%) | est. treated pixels {st['n_treated_pixels_est']:,} | est. control pixels {st['n_control_pixels_est']:,}")
    print(f"treatment x post cells: {st['cells_treat_x_post']}")
    (P.ok if st['duplicates_total']==0 else P.fail)(f"duplicate pixel-Year-Season rows: {st['duplicates_total']} (must be 0)")
    pct=st['pct_pixels_multi_year']; (P.ok if pct>50 else P.warn)(f"{pct}% of pixels appear in >1 year; {st['pct_pixels_6plus_years']}% in >=6 years")
    if pct<=50: P.info("LOW persistence -> check UID_PRECISION_DEC (5 for a 10 m grid) and whether tiles/shapefile changed between years")
    for k in ["buff_km","Season","treatment","control","pre","post","did_term"]:
        if k in st["unique_key_values"]: print(f"  {k:10s} -> {dict(sorted(st['unique_key_values'][k].items(),key=lambda x:str(x[0])))}")
    ncl=len(st["unique_key_values"].get("subwshed_id",{})) or None
    npre=sum(1 for y in st["unique_key_values"].get("Year",{}) if int(y)<P.POST_CUTOFF)
    V=P.panel_validity_from_stats(st,n_clusters=ncl,n_pre_years=npre)
    for f in V["failures"]: P.fail(f)
    for w in V["warnings"]: P.warn(w)
    (P.ok if V["VERDICT"].startswith("VALID") else P.fail)(f"VERDICT: {V['VERDICT']}")
    verdict={"the four models (M01 TWFE, M02 event study, M16 joint pre-trends test, M34 Honest DiD sensitivity)":V["VERDICT"].startswith("VALID")}
    print()
    for k,v in verdict.items(): (P.ok if v else P.warn)(f"{'CAN RUN ' if v else 'BLOCKED  '} {k}")
    json.dump({"stats":st,"validity":V,"can_run":{k:bool(v) for k,v in verdict.items()}},
              open(os.path.join(P.OUTPUT_DIR,"readiness_report.json"),"w"),indent=2,default=str)
    P.ok("readiness_report.json saved")
    # v20.58: duplicates CONFIRMED on the finished panel (never assumed) and the outcomes that are the same variable (LSWI = NDMI,
    # WSSI = 1 - ESI in your exports) -- said here once, and at every model run of either one
    P.confirm_panel_duplicates(P.FINAL_PANEL)
    import _common as _Cid; _Cid.PREPARED_PANEL = P.FINAL_PANEL; _Cid.outcome_identities(verbose=True)
except Exception as e:
    P.fail(f"STEP 6 failed: {e}"); traceback.print_exc(); raise
P.done(6, next_task="STEP 8 -- the design defaults the four models inherit")

## P00_Design_for_Every_Model
the design every model inherits (saved scenario), season report  
*(formerly `P00_Design_for_Every_Model.ipynb`)*

In [ ]:
# ===== P00_Design_for_Every_Model.ipynb (merged into P00) =====
# v17.2 auto-bootstrap: this cell can be run on its own (finds _common.py / _prep_common.py)
if "P" not in globals() or "C" not in globals():
    import os as _os, sys as _sys, importlib as _il
    _d = _os.path.abspath(_os.getcwd()); _found = None
    for _ in range(6):
        for _c in [_d] + [_p for _p in __import__("glob").glob(_os.path.join(_d, "*")) if _os.path.isdir(_p)]:
            if _os.path.exists(_os.path.join(_c, "_common.py")): _found = _c; break
        if _found or _os.path.dirname(_d) == _d: break
        _d = _os.path.dirname(_d)
    if _found is None: raise RuntimeError("_common.py not found -- run CELL 1 (import bootstrap) first")
    if _found not in _sys.path: _sys.path.insert(0, _found)
    P = _il.import_module("_prep_common"); C = _il.import_module("_common")
import os
P.step(8, "the design saved for every model (the defaults for what a model's CELL 1 leaves unset), the timing per sub-watershed, the season report")
# v20.57: the DESIGN is chosen in each MODEL, when it runs (its CELL 1: timing, rings, years, seasons, fragments, dose, fixed effects);
# nothing here has to be re-run for it. This step writes P00's reports and the DEFAULT scenario (did_scenario.json) a model adopts
# only for an option its CELL 1 leaves unset (None). The same resolve_design() the models call is used, so the defaults saved here
# are exactly what a model would use with the same settings.
try:
    rep = C.season_choice_report(path=P.FINAL_PANEL, save_as=os.path.join(P.OUTPUT_DIR, "season_choice_report.csv"))
    P.ok(f"season_choice_report.csv written -> {P.OUTPUT_DIR}")
    C.set_scenario(design_mode=DESIGN_MODE, timing=TREATMENT_TIMING, treatment_year=TREATMENT_YEAR, fund_start_rule=FUND_START_RULE,
                   control_zones=CONTROL_ZONES, pre_years=PRE_YEARS, post_years=POST_YEARS, seasons=SEASONS,
                   exclude_transition_year=EXCLUDE_TRANSITION_YEAR, cohort_offset=COHORT_OFFSET, cluster=CLUSTER, pooled_fe=POOLED_FE,
                   unit_fe=UNIT_FE, overlap_rows=OVERLAP_ROWS, fragment_rule=FRAGMENT_RULE, exclude_gapfilled=EXCLUDE_GAPFILLED, verbose=False)
    C.resolve_design(verbose=True, force=True)          # fragments -> the sub-watersheds of this panel -> their timing -> the "data" options
    C.fragment_table(verbose=True)                      # what the fragment rule leaves out (FRAGMENTS_TABLE.json next to the panel)
    _tt = C.timing_table(save_as=os.path.join(C.RESULTS_ROOT, "sws_timing.csv"))
    print(_tt.to_string(index=False))
    _n = int((_tt["site_id"] > 0).sum())
    C.info(("ONE sub-watershed in this panel after the fragment rule -> single design (inference on years)" if _n <= 1 else
            f"{_n} sub-watersheds in this panel -> POOLED design: each one on its own timing (its cohort), a cluster once there are >= 6; "
            + ("sub-watershed x period fixed effects" if POOLED_FE == "site_period" else "one period fixed effect shared by all (POOLED_FE = 'period')")))
    C.save_scenario(verbose=True)      # the DEFAULTS for an option a model's CELL 1 leaves unset
except Exception as e:
    P.fail(f"STEP 8 failed: {e}"); import traceback; traceback.print_exc(); raise
P.done(8, next_task="P09 (estimator files) -> P10 (whether this data supports the four models) -> P12 (their packages)")


## P09_Estimator_Variable_Files
per-variable files, baselines and pixel counts of every variable  
*(formerly `P09_Estimator_Variable_Files.ipynb`)*

In [ ]:
# ===== P09_Estimator_Variable_Files.ipynb (merged into P00) =====
if RUN_ESTIMATOR_FILES:
    import _common as C, pandas as pd
    C.step(9, "per-variable estimator files + baselines and pixel counts of every variable")
    C.autotune()
    manifest = C.cell_guard(C.build_estimator_files)(outcomes=OUTCOMES, overwrite=OVERWRITE)
    if manifest:
        print(pd.DataFrame(manifest)[["outcome", "rows", "bytes_on_disk"]].to_string(index=False))
    C.baseline_and_counts_all(outcomes=OUTCOMES)
    C.screen_all_outcomes(outcomes=OUTCOMES)       # v20.45: fill / collapsed year-seasons per variable -> OUTCOME_SCREEN.md     # results\BASELINE\<scenario>\BASELINE_MEANS_ALL_VARIABLES.csv / .txt
    # v20.40: readiness is reported ONCE, by P10 below (complete / limited / incomplete-need data + PANEL-WIDE CAVEATS)
    C.done(9, next_task="P10_Model_Readiness")
else:
    print("[INFO]    P09_Estimator_Variable_Files skipped (RUN_ESTIMATOR_FILES = False in P00_Settings)")


## P10_Model_Readiness
whether this panel completes / limits / needs data for each of the four models  
*(formerly `P10_Model_Readiness.ipynb`)*

In [ ]:
# ===== P10_Model_Readiness.ipynb (merged into P00) =====
if RUN_READINESS:
    import _common as C, _readiness as R
    C.load_scenario(verbose=False)
    _outs = OUTCOMES or (C.SELECTED_OUTCOMES if getattr(C, "SELECTED_OUTCOMES", None) else ["NDVI"])
    table, ctx = R.model_readiness(C=C, outcomes=_outs, compare="auto")   # one sub-watershed -> + the 20-sub-watershed projection
else:
    print("[INFO]    P10_Model_Readiness skipped (RUN_READINESS = False in P00_Settings)")


## P12_Prebuilt_Packages_Verify
Install the packages of the four models' pre-built routes (Python: pyfixest; R: fixest, HonestDiD), verify each route on a known answer, and report what computes each of the four models now (PREBUILT_READINESS.md): a verified package first, the engine only when none is available.  
*(formerly `P12_Prebuilt_Packages_Verify.ipynb`)*

In [ ]:
# ===== P12_Prebuilt_Packages_Verify.ipynb (merged into P00) =====
if VERIFY_PACKAGES:
    import _common as C, importlib
    C.AUTO_INSTALL_PACKAGES = bool(INSTALL_PACKAGES); C.R_SCRIPT = R_SCRIPT
    # 1. v20.55: EVERY package of this project -- the four models' (Python: pyfixest; R: fixest, HonestDiD; _paths.PIPELINE_MODELS) -- checked, installed when missing (Python: pre-built wheel -> source -> local wheel
    #    folder; R: CRAN binary -> source -> author's r-universe -> GitHub -> archive -> mirror) and CONFIRMED: "N of N installed"
    #    (results/PACKAGE_STATUS.csv names each one, its version, the route that provided it and the models it serves)
    _st = C.confirm_packages(install=bool(INSTALL_PACKAGES), r=bool(INSTALL_R_PACKAGES) or bool(C.find_rscript()))
    _new_py = _st[(_st.language == "Python") & ~_st.how.isin(["installed", "NOT installed"])]
    if len(_new_py): C.warn(f"Python packages installed in this kernel ({', '.join(_new_py.package)}) -- RESTART the kernel once, then re-run P00 from P00_Settings, so they load cleanly")
    # 1b. v20.58: the OUT-OF-CORE engines (used only beyond 98 % of the RAM; OUT_OF_CORE in _paths.py): Dask -> Spark -> built-in batches
    try:
        import _outofcore as _O; _O.status_table()
    except Exception as _e:
        C.warn(f"out-of-core engines: {type(_e).__name__}: {_e}")
    _miss = _st[~_st.installed & (_st.kind != "optional")]
    if len(_miss) and not INSTALL_PACKAGES:
        C.warn(f"not installed: {list(_miss.package)} -- set INSTALL_PACKAGES = True (internet), or offline: pip download <pkg> -d python_prebuilt/wheels "
               f"on a connected machine (the chain installs from that folder)")
    # 2. every route checked on a KNOWN answer before it is trusted (verdicts per package version -> prebuilt_verified.json)
    ver_f = C.verify_prebuilt_routes()             # function layer: pyfixest TWFE / event study / wild bootstrap, diff-diff CS
    ver_m = C.verify_prebuilt_models()             # model layer: diff-diff, econml / doubleml, esda
    # R: the second choice (models_prebuilt.R through R/run_one.R) -- found, installed, verified like the Python routes
    C.R_SCRIPT = R_SCRIPT
    if C.find_rscript():
        C.info(f"R found: {C.find_rscript()} (R {C.r_versions().get('R')})")
        ver_r = C.verify_r_models()                # every R route on the same known answers (the R packages were installed / confirmed above)
    else:
        C.warn("R not found -- the R routes stay off. Install R (https://cran.r-project.org) or set R_SCRIPT in P00_Settings")
    C.PREBUILT_MODE = "auto"                       # verified Python package first, then verified R package, then the existing implementation
    # 3. the four models: which package computes each NOW, why not when not, and whether the panel holds its inputs
    ready = C.prebuilt_readiness()                 # -> results\PREBUILT_READINESS.csv / .md
    print(ready[["model", "route", "verified", "r_route", "r_verified", "computes_now", "panel_inputs"]].to_string(index=False))
else:
    print("[INFO]    P12_Prebuilt_Packages_Verify skipped (VERIFY_PACKAGES = False in P00_Settings)")


---
**P00 complete.** Next: the four models, one at a time — `02_Core_DiD_Models/M01…`, `M02…`, `M16…` and `04_Advanced_Staggered_Robustness/M34…` — each with its design (timing, rings, years, seasons, covariates) in its CELL 1; `08_Multisite_Runs/MS01_Multisite_Runs.ipynb` for per-site + pooled runs of M01; `06_Validation/V01_Results_Audit.ipynb` to audit every result.